# SQL analysis — October 19, 2026
Live PostgreSQL queries ran in a read-only transaction. These saved outputs describe current constituents and current sector assignments. Missing values remain missing.
Schema and keys matched the repository. 503 current companies; 2,455 financial rows covering 500 tickers. Latest views contain 401 leverage values and 96 interest-coverage values. HONA, VYLR and XOM have no financial rows.
The SQL below is displayed for inspection; results are the saved snapshot, not an automatic live refresh.

## historical_trends
```sql
-- Current constituents and current sectors, grouped by calendar year of fiscal-period end.
-- Keep one latest period per ticker/year so shortened periods do not double-weight a company.
-- This is a changing observed sample, not a matched cohort or historical index membership.
WITH annual AS (
    SELECT DISTINCT ON (f.ticker, EXTRACT(YEAR FROM f.fiscal_period_end))
        c.sector,
        f.ticker,
        EXTRACT(YEAR FROM f.fiscal_period_end)::INTEGER AS period_end_year,
        f.debt_to_assets_ratio,
        f.interest_coverage_ratio
    FROM financials f
    JOIN companies c ON c.ticker = f.ticker
    WHERE c.is_current_constituent = TRUE
    ORDER BY f.ticker, EXTRACT(YEAR FROM f.fiscal_period_end), f.fiscal_period_end DESC
)
SELECT
    sector,
    period_end_year,
    COUNT(*) AS observed_companies,
    COUNT(debt_to_assets_ratio) AS leverage_available,
    COUNT(interest_coverage_ratio) AS interest_coverage_available,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY debt_to_assets_ratio) AS median_liabilities_to_assets,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY interest_coverage_ratio) AS median_interest_coverage
FROM annual
GROUP BY sector, period_end_year
ORDER BY sector, period_end_year;
```


In [ ]:
from pathlib import Path
import json
root = Path.cwd()
if root.name == 'research':
    root = root.parent.parent
json.loads((root / 'scratch/research/sql_analysis_2026-10-19.json').read_text())['historical_trends']

## missing_coverage
```sql
-- Distinguish no financial row from a missing ratio in the latest stored row.
SELECT
    sector,
    COUNT(*) AS current_companies,
    COUNT(*) FILTER (WHERE fiscal_period_end IS NULL) AS no_financial_history,
    COUNT(*) FILTER (WHERE fiscal_period_end IS NOT NULL AND debt_to_assets_ratio IS NULL) AS latest_leverage_missing,
    COUNT(*) FILTER (WHERE fiscal_period_end IS NOT NULL AND interest_coverage_ratio IS NULL) AS latest_interest_coverage_missing,
    ROUND(100.0 * COUNT(*) FILTER (WHERE debt_to_assets_ratio IS NULL) / COUNT(*), 2) AS leverage_missing_pct,
    ROUND(100.0 * COUNT(*) FILTER (WHERE interest_coverage_ratio IS NULL) / COUNT(*), 2) AS interest_coverage_missing_pct
FROM sector_leverage_rankings
GROUP BY sector
ORDER BY interest_coverage_missing_pct DESC, sector;
```


In [ ]:
from pathlib import Path
import json
root = Path.cwd()
if root.name == 'research':
    root = root.parent.parent
json.loads((root / 'scratch/research/sql_analysis_2026-10-19.json').read_text())['missing_coverage']

## outliers
```sql
-- Two-sided sector IQR flags are review candidates, not errors or distress predictions.
-- Require at least eight observed companies and a nonzero IQR for each metric.
WITH metrics AS (
    SELECT ticker, company_name, sector, fiscal_period_end,
        'liabilities_to_assets' AS metric, debt_to_assets_ratio AS value
    FROM sector_leverage_rankings
    UNION ALL
    SELECT ticker, company_name, sector, fiscal_period_end,
        'interest_coverage' AS metric, interest_coverage_ratio AS value
    FROM sector_interest_coverage_rankings
), quartiles AS (
    SELECT sector, metric, COUNT(value) AS observed_companies,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY value) AS q1,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY value) AS q3
    FROM metrics
    GROUP BY sector, metric
)
SELECT m.*, q.observed_companies,
    q.q1 - 1.5 * (q.q3 - q.q1) AS lower_fence,
    q.q3 + 1.5 * (q.q3 - q.q1) AS upper_fence
FROM metrics m
JOIN quartiles q ON q.sector = m.sector AND q.metric = m.metric
WHERE q.observed_companies >= 8 AND q.q3 > q.q1
    AND (m.value < q.q1 - 1.5 * (q.q3 - q.q1)
         OR m.value > q.q3 + 1.5 * (q.q3 - q.q1))
ORDER BY m.metric, m.sector, m.value DESC;
```


In [ ]:
from pathlib import Path
import json
root = Path.cwd()
if root.name == 'research':
    root = root.parent.parent
json.loads((root / 'scratch/research/sql_analysis_2026-10-19.json').read_text())['outliers']

## sector_comparison
```sql
-- Latest stored period per current constituent; NULL never falls back to an older value.
-- Medians describe reporting companies, not the full sector or an asset-weighted ratio.
SELECT
    sector,
    COUNT(*) AS current_companies,
    COUNT(fiscal_period_end) AS companies_with_financials,
    MIN(fiscal_period_end) AS earliest_latest_period,
    MAX(fiscal_period_end) AS latest_period,
    COUNT(debt_to_assets_ratio) AS leverage_available,
    ROUND(100.0 * COUNT(debt_to_assets_ratio) / COUNT(*), 2) AS leverage_coverage_pct,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY debt_to_assets_ratio) AS median_liabilities_to_assets,
    COUNT(interest_coverage_ratio) AS interest_coverage_available,
    ROUND(100.0 * COUNT(interest_coverage_ratio) / COUNT(*), 2) AS interest_coverage_pct,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY interest_coverage_ratio) AS median_interest_coverage
FROM sector_leverage_rankings
GROUP BY sector
ORDER BY sector;
```


In [ ]:
from pathlib import Path
import json
root = Path.cwd()
if root.name == 'research':
    root = root.parent.parent
json.loads((root / 'scratch/research/sql_analysis_2026-10-19.json').read_text())['sector_comparison']